# AI 서비스 임원보고용 분석 노트북 (간결 시각화 버전)

- 목적: 임원 요청 9개 항목을 표 중심으로 빠르게 점검
- 원칙:
  - 시각화는 꼭 필요한 핵심 추이만 최소 사용
  - 결과표는 print 시 마크다운 표 형태로 바로 복붙 가능
- 오픈일 기준 코호트 방식으로 재사용/전후 분석 처리

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid")

def safe_div(n, d):
    if d is None or d == 0 or pd.isna(d):
        return np.nan
    return n / d

def pct_change(new, old):
    if old is None or old == 0 or pd.isna(old):
        return np.nan
    return (new - old) / old

def print_table(title, df, digits=4):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)
    if df is None or len(df) == 0:
        print("(데이터 없음)")
        return
    show = df.copy()
    num_cols = show.select_dtypes(include=["number"]).columns
    for c in num_cols:
        show[c] = show[c].round(digits)
    try:
        print(show.to_markdown(index=False))
    except Exception:
        print(show.to_string(index=False))

In [ ]:
# 경로 설정 (필요시 파일명 변경)
OPEN_DATE = pd.Timestamp("2026-03-23")
DATA_DIR = Path("data")

daily_file = DATA_DIR / "daily_metrics.csv"
profile_file = DATA_DIR / "customer_profile.csv"
chat_file = DATA_DIR / "ai_chat_daily_by_user.csv"
ai_transfer_file = DATA_DIR / "ai_transfer_daily_by_user.csv"
event_file = DATA_DIR / "event_calendar.csv"
banner_file = DATA_DIR / "banner_funnel.csv"

In [ ]:
def load_csv(path):
    if path.exists():
        return pd.read_csv(path)
    print(f"[WARN] 파일 없음: {path}")
    return pd.DataFrame()

def first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def normalize_daily(df):
    if df.empty:
        return df
    mapper = {
        "date": ["date", "일자", "날짜"],
        "app_logins": ["app_logins", "일일 앱 로그인고객수", "login_users"],
        "new_signups": ["new_signups", "일일 신규가입자", "전자금융신규가입자"],
        "ai_signups": ["ai_signups", "일일 ai 가입자", "daily_ai_signups"],
    }
    rename = {}
    for std, cands in mapper.items():
        found = first_existing_column(df, cands)
        if found:
            rename[found] = std
    out = df.rename(columns=rename).copy()
    if "date" in out.columns:
        out["date"] = pd.to_datetime(out["date"], errors="coerce")
    return out

def normalize_profile(df):
    if df.empty:
        return df
    mapper = {
        "customer_id": ["customer_id", "고객번호"],
        "age_band": ["age_band", "나이대"],
        "is_employee": ["is_employee", "임직원여부"],
        "ebank_signup_date": ["ebank_signup_date", "전자금융가입일", "first_account_open_date"],
        "ai_signup_date": ["ai_signup_date", "AI가입일", "ai_join_date"],
        "last_login_date": ["last_login_date", "최근접속일"],
        "pre30_transfer_count": ["pre30_transfer_count", "AI가입전30일_이체건수"],
        "post30_ai_transfer_count": ["post30_ai_transfer_count", "AI가입후30일_ai이체건수"],
        "post30_other_transfer_count": ["post30_other_transfer_count", "AI가입후30일_기존이체건수"],
        "pre_year_avg_transfer_count": ["pre_year_avg_transfer_count", "AI가입전1년_월평균이체건수"],
        "feedback_like_count": ["feedback_like_count", "피드백좋아요건"],
        "feedback_dislike_count": ["feedback_dislike_count", "피드백싫어요건"],
        "unanswered_count": ["unanswered_count", "답변불가경험건수"],
    }
    rename = {}
    for std, cands in mapper.items():
        found = first_existing_column(df, cands)
        if found:
            rename[found] = std
    out = df.rename(columns=rename).copy()
    for c in ["ebank_signup_date", "ai_signup_date", "last_login_date"]:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c], errors="coerce")
    return out

def normalize_chat(df):
    if df.empty:
        return df
    mapper = {
        "customer_id": ["customer_id", "고객번호"],
        "ai_signup_date": ["ai_signup_date", "AI가입일"],
        "chat_date": ["chat_date", "채팅요청일", "요청일", "date"],
        "service_category": ["service_category", "서비스분류", "서비스분류코드"],
        "request_count": ["request_count", "건수", "cnt"],
    }
    rename = {}
    for std, cands in mapper.items():
        found = first_existing_column(df, cands)
        if found:
            rename[found] = std
    out = df.rename(columns=rename).copy()
    for c in ["ai_signup_date", "chat_date"]:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c], errors="coerce")
    return out

def normalize_ai_transfer(df):
    if df.empty:
        return df
    mapper = {
        "customer_id": ["customer_id", "고객번호"],
        "ai_signup_date": ["ai_signup_date", "AI가입일"],
        "ai_transfer_date": ["ai_transfer_date", "ai이체일"],
        "ai_transfer_count": ["ai_transfer_count", "ai이체건수"],
        "ai_transfer_amount": ["ai_transfer_amount", "금액합"],
    }
    rename = {}
    for std, cands in mapper.items():
        found = first_existing_column(df, cands)
        if found:
            rename[found] = std
    out = df.rename(columns=rename).copy()
    for c in ["ai_signup_date", "ai_transfer_date"]:
        if c in out.columns:
            out[c] = pd.to_datetime(out[c], errors="coerce")
    return out

def normalize_event(df):
    if df.empty:
        return df
    mapper = {
        "event_date": ["event_date", "일자", "날짜"],
        "event_name": ["event_name", "이벤트명", "event"],
        "event_type": ["event_type", "이벤트유형", "type"],
    }
    rename = {}
    for std, cands in mapper.items():
        found = first_existing_column(df, cands)
        if found:
            rename[found] = std
    out = df.rename(columns=rename).copy()
    if "event_date" in out.columns:
        out["event_date"] = pd.to_datetime(out["event_date"], errors="coerce")
    return out

def normalize_banner(df):
    if df.empty:
        return df
    mapper = {
        "date": ["date", "일자", "날짜"],
        "banner_name": ["banner_name", "배너위치", "배너명"],
        "banner_clicks": ["banner_clicks", "클릭수", "배너클릭수"],
        "banner_signups": ["banner_signups", "가입자수", "클릭후가입자수"],
    }
    rename = {}
    for std, cands in mapper.items():
        found = first_existing_column(df, cands)
        if found:
            rename[found] = std
    out = df.rename(columns=rename).copy()
    if "date" in out.columns:
        out["date"] = pd.to_datetime(out["date"], errors="coerce")
    return out

def pre_post_28(df, metric_col):
    if df.empty or metric_col not in df.columns:
        return pd.DataFrame()
    pre = df[(df["date"] >= OPEN_DATE - pd.Timedelta(days=28)) & (df["date"] < OPEN_DATE)]
    post = df[(df["date"] >= OPEN_DATE) & (df["date"] < OPEN_DATE + pd.Timedelta(days=28))]
    return pd.DataFrame([
        {
            "metric": metric_col,
            "pre_days": len(pre),
            "post_days": len(post),
            "pre_mean": pre[metric_col].mean(),
            "post_mean": post[metric_col].mean(),
            "pct_change": pct_change(post[metric_col].mean(), pre[metric_col].mean()),
        }
    ])

In [ ]:
daily = normalize_daily(load_csv(daily_file))
profile = normalize_profile(load_csv(profile_file))
chat = normalize_chat(load_csv(chat_file))
ai_transfer = normalize_ai_transfer(load_csv(ai_transfer_file))
events = normalize_event(load_csv(event_file))
banner = normalize_banner(load_csv(banner_file))

print_table("데이터 로딩 현황", pd.DataFrame([
    {"dataset": "daily", "rows": len(daily)},
    {"dataset": "profile", "rows": len(profile)},
    {"dataset": "chat", "rows": len(chat)},
    {"dataset": "ai_transfer", "rows": len(ai_transfer)},
    {"dataset": "events", "rows": len(events)},
    {"dataset": "banner", "rows": len(banner)},
]))

In [ ]:
# 1) 오픈 전후 로그인, 2) 신규가입, 3) AI가입 추이
t1 = pre_post_28(daily, "app_logins")
t2 = pre_post_28(daily, "new_signups")
t3 = pre_post_28(daily, "ai_signups")

if not daily.empty and {"app_logins", "ai_signups"}.issubset(daily.columns):
    daily["ai_signup_per_1000_login"] = daily.apply(
        lambda r: safe_div(r["ai_signups"], r["app_logins"]) * 1000, axis=1
    )
    t1b = pre_post_28(daily, "ai_signup_per_1000_login")
else:
    t1b = pd.DataFrame()

print_table("[1] 오픈 전후 로그인 추이", t1)
print_table("[1-보완] 로그인 1천명당 AI가입", t1b)
print_table("[2] 오픈 전후 신규가입 추이", t2)
print_table("[3] 오픈 전후 AI가입 추이", t3)

# 필요한 핵심 그래프만: 2개
if not daily.empty and "date" in daily.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    if "app_logins" in daily.columns:
        sns.lineplot(data=daily.sort_values("date"), x="date", y="app_logins", ax=axes[0], label="로그인")
    if "new_signups" in daily.columns:
        sns.lineplot(data=daily.sort_values("date"), x="date", y="new_signups", ax=axes[0], label="신규가입")
    axes[0].axvline(OPEN_DATE, color="red", linestyle="--", label="오픈일")
    axes[0].set_title("로그인/신규가입 추이")
    axes[0].legend()

    if "ai_signups" in daily.columns:
        sns.lineplot(data=daily.sort_values("date"), x="date", y="ai_signups", ax=axes[1], label="AI가입")
    axes[1].axvline(OPEN_DATE, color="red", linestyle="--", label="오픈일")
    if not events.empty and "event_date" in events.columns:
        for _, r in events.dropna(subset=["event_date"]).iterrows():
            axes[1].axvline(r["event_date"], color="gray", linestyle=":", alpha=0.5)
    axes[1].set_title("AI가입 추이 (이벤트 기준선 포함)")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

# 3-보완) 이벤트 전후 7일 비교
event_rows = []
if not daily.empty and not events.empty and {"date", "ai_signups"}.issubset(daily.columns) and "event_date" in events.columns:
    for _, r in events.dropna(subset=["event_date"]).iterrows():
        dt = r["event_date"]
        pre7 = daily[(daily["date"] >= dt - pd.Timedelta(days=7)) & (daily["date"] < dt)]["ai_signups"].mean()
        post7 = daily[(daily["date"] > dt) & (daily["date"] <= dt + pd.Timedelta(days=7))]["ai_signups"].mean()
        tail = daily[(daily["date"] > dt + pd.Timedelta(days=7)) & (daily["date"] <= dt + pd.Timedelta(days=14))]["ai_signups"].mean()
        d1 = pct_change(post7, pre7)
        d2 = pct_change(tail, pre7)
        if pd.notna(d1) and d1 >= 0.10 and pd.notna(d2) and d2 >= 0.05:
            impact = "지속 증가"
        elif pd.notna(d1) and d1 >= 0.05:
            impact = "단기 반응"
        else:
            impact = "유의 변화 제한"
        event_rows.append({
            "event_name": r["event_name"] if "event_name" in events.columns else str(dt.date()),
            "event_date": dt.date(),
            "pre7_mean": pre7,
            "post7_mean": post7,
            "post7_pct_change": d1,
            "post8_14_pct_change": d2,
            "impact_type": impact,
        })
event_df = pd.DataFrame(event_rows)
print_table("[3-보완] 이벤트 반응 유형", event_df)

In [ ]:
# 4) 배너 클릭 후 가입 전환율
if not banner.empty and {"banner_clicks", "banner_signups"}.issubset(banner.columns):
    gcols = ["banner_name"] if "banner_name" in banner.columns else []
    if gcols:
        t4 = banner.groupby(gcols, dropna=False).agg(clicks=("banner_clicks", "sum"), signups=("banner_signups", "sum")).reset_index()
    else:
        t4 = pd.DataFrame([{"banner_name": "전체", "clicks": banner["banner_clicks"].sum(), "signups": banner["banner_signups"].sum()}])
    t4["click_to_signup_cvr"] = t4.apply(lambda r: safe_div(r["signups"], r["clicks"]), axis=1)
else:
    t4 = pd.DataFrame()
print_table("[4] 배너 클릭→가입 전환율", t4)

# 5) AI가입일 = 신규가입일 동일 여부
if not profile.empty and {"ebank_signup_date", "ai_signup_date"}.issubset(profile.columns):
    p = profile.dropna(subset=["ebank_signup_date", "ai_signup_date"]).copy()
    p["diff_days"] = (p["ai_signup_date"] - p["ebank_signup_date"]).dt.days
    t5 = pd.DataFrame([
        {"group": "동일일(0일)", "users": int((p["diff_days"] == 0).sum())},
        {"group": "1~7일", "users": int(((p["diff_days"] >= 1) & (p["diff_days"] <= 7)).sum())},
        {"group": "8~30일", "users": int(((p["diff_days"] >= 8) & (p["diff_days"] <= 30)).sum())},
        {"group": "30일 초과", "users": int((p["diff_days"] > 30).sum())},
    ])
    t5["ratio"] = t5["users"] / max(t5["users"].sum(), 1)
else:
    t5 = pd.DataFrame()
print_table("[5] AI가입일 vs 신규가입일 간격", t5)

In [ ]:
# 6) AI가입 전후 이체 양상 + 퍼널, 7) 재사용, 8) 미사용 분류

if not chat.empty and {"customer_id", "ai_signup_date", "chat_date", "request_count"}.issubset(chat.columns):
    cc = chat.dropna(subset=["customer_id", "ai_signup_date", "chat_date"]).copy()
    cc = cc[cc["chat_date"] >= cc["ai_signup_date"]]
    cc["day_offset"] = (cc["chat_date"] - cc["ai_signup_date"]).dt.days
    usage_user = cc.groupby("customer_id").agg(
        total_requests=("request_count", "sum"),
        use_days=("chat_date", "nunique"),
        req_30d=("request_count", lambda s: s[cc.loc[s.index, "day_offset"] <= 30].sum()),
        days_30d=("chat_date", lambda s: cc.loc[s.index][cc.loc[s.index, "day_offset"] <= 30]["chat_date"].nunique()),
        last_use_date=("chat_date", "max"),
    ).reset_index()
else:
    usage_user = pd.DataFrame(columns=["customer_id", "total_requests", "use_days", "req_30d", "days_30d", "last_use_date"])

# 6-a 전후 이체 요약
if not profile.empty:
    for c in ["pre30_transfer_count", "post30_ai_transfer_count", "post30_other_transfer_count"]:
        if c not in profile.columns:
            profile[c] = np.nan
    profile["post30_total_transfer_count"] = profile["post30_ai_transfer_count"].fillna(0) + profile["post30_other_transfer_count"].fillna(0)
    t6a = pd.DataFrame([
        {"metric": "가입 전 30일 평균 이체건수", "value": profile["pre30_transfer_count"].mean()},
        {"metric": "가입 후 30일 평균 전체 이체건수", "value": profile["post30_total_transfer_count"].mean()},
        {"metric": "가입 후 30일 평균 AI이체건수", "value": profile["post30_ai_transfer_count"].mean()},
        {"metric": "가입 후 30일 평균 기존이체건수", "value": profile["post30_other_transfer_count"].mean()},
    ])
else:
    t6a = pd.DataFrame()

# 6-b 실행 퍼널
signup_users = profile["customer_id"].nunique() if (not profile.empty and "customer_id" in profile.columns) else np.nan
req_users = usage_user["customer_id"].nunique() if not usage_user.empty else np.nan

if not chat.empty and {"customer_id", "service_category"}.issubset(chat.columns):
    transfer_req_users = chat[chat["service_category"].astype(str).str.contains("이체|transfer", case=False, na=False)]["customer_id"].nunique()
else:
    transfer_req_users = np.nan

if not ai_transfer.empty and {"customer_id", "ai_transfer_count"}.issubset(ai_transfer.columns):
    transfer_exec_users = ai_transfer[ai_transfer["ai_transfer_count"] > 0]["customer_id"].nunique()
else:
    transfer_exec_users = np.nan

t6b = pd.DataFrame([
    {"stage": "AI가입자", "users": signup_users, "ratio_vs_signup": 1.0},
    {"stage": "AI요청 경험자", "users": req_users, "ratio_vs_signup": safe_div(req_users, signup_users)},
    {"stage": "AI이체 요청 경험자", "users": transfer_req_users, "ratio_vs_signup": safe_div(transfer_req_users, signup_users)},
    {"stage": "AI이체 실제 실행자", "users": transfer_exec_users, "ratio_vs_signup": safe_div(transfer_exec_users, signup_users)},
])

print_table("[6] 가입 전후 이체 양상", t6a)
print_table("[6] AI이체 퍼널", t6b)

# 7 재사용 지표
if not usage_user.empty:
    usage_user["is_reuser"] = usage_user["use_days"] >= 2
    usage_user["is_30d_2plus"] = usage_user["req_30d"] >= 2
    usage_user["is_30d_3days"] = usage_user["days_30d"] >= 3
    base = signup_users if pd.notna(signup_users) else usage_user["customer_id"].nunique()
    t7 = pd.DataFrame([
        {"metric": "가입자수", "users": base, "ratio_vs_signup": 1.0},
        {"metric": "가입 후 1회 이상 사용자", "users": usage_user["customer_id"].nunique(), "ratio_vs_signup": safe_div(usage_user["customer_id"].nunique(), base)},
        {"metric": "가입 후 재사용자(2일 이상)", "users": int(usage_user["is_reuser"].sum()), "ratio_vs_signup": safe_div(int(usage_user["is_reuser"].sum()), base)},
        {"metric": "가입 후 30일 내 2회 이상", "users": int(usage_user["is_30d_2plus"].sum()), "ratio_vs_signup": safe_div(int(usage_user["is_30d_2plus"].sum()), base)},
        {"metric": "가입 후 30일 내 3일 이상", "users": int(usage_user["is_30d_3days"].sum()), "ratio_vs_signup": safe_div(int(usage_user["is_30d_3days"].sum()), base)},
    ])
else:
    t7 = pd.DataFrame()
print_table("[7] AI 재사용 지표", t7)

# 8 미사용/1회성/재사용 분류
if not profile.empty and "customer_id" in profile.columns:
    seg = profile[["customer_id", "ai_signup_date"]].dropna(subset=["ai_signup_date"]).drop_duplicates().copy()
    seg = seg.merge(usage_user[["customer_id", "total_requests", "use_days", "last_use_date"]], on="customer_id", how="left")
    max_chat_date = chat["chat_date"].max() if (not chat.empty and "chat_date" in chat.columns) else pd.Timestamp.today().normalize()
    def assign_segment(r):
        req = r.get("total_requests", 0)
        days = r.get("use_days", 0)
        last = r.get("last_use_date", pd.NaT)
        if pd.isna(req) or req <= 0 or pd.isna(days) or days <= 0:
            return "미사용"
        if days == 1:
            return "1회성"
        if pd.notna(last) and last < (max_chat_date - pd.Timedelta(days=30)):
            return "휴면(최근30일 미사용)"
        return "재사용"
    seg["usage_segment"] = seg.apply(assign_segment, axis=1)
    t8 = seg.groupby("usage_segment")["customer_id"].nunique().reset_index(name="users")
    t8["ratio"] = t8["users"] / max(t8["users"].sum(), 1)
else:
    seg = pd.DataFrame()
    t8 = pd.DataFrame()
print_table("[8] AI가입 후 미사용/1회성/재사용", t8)

In [ ]:
# 9) 미사용자 vs 재사용자 특성 비교
if not profile.empty and not seg.empty:
    p9 = profile.merge(seg[["customer_id", "usage_segment"]], on="customer_id", how="left")
    p9["usage_segment"] = p9["usage_segment"].fillna("미분류")

    numeric_features = [
        "pre_year_avg_transfer_count",
        "pre30_transfer_count",
        "post30_ai_transfer_count",
        "feedback_like_count",
        "feedback_dislike_count",
        "unanswered_count",
    ]
    numeric_features = [c for c in numeric_features if c in p9.columns]

    rows = []
    for col in numeric_features:
        g = p9.groupby("usage_segment")[col].mean().reset_index()
        for _, r in g.iterrows():
            rows.append({"feature": col, "usage_segment": r["usage_segment"], "mean_value": r[col]})
    t9_num = pd.DataFrame(rows)

    cat_rows = []
    for c in ["age_band", "is_employee"]:
        if c in p9.columns:
            g = p9.groupby(["usage_segment", c])["customer_id"].nunique().reset_index(name="users")
            g["ratio_in_segment"] = g.groupby("usage_segment")["users"].transform(lambda s: s / s.sum())
            g["feature"] = c
            cat_rows.append(g)
    t9_cat = pd.concat(cat_rows, ignore_index=True) if len(cat_rows) else pd.DataFrame()
else:
    t9_num = pd.DataFrame()
    t9_cat = pd.DataFrame()

print_table("[9] 특성비교 - 수치형", t9_num)
print_table("[9] 특성비교 - 범주형", t9_cat)

# 임원 보고용 문구 템플릿(결과가 좋거나 나쁠 때)
print("\n" + "="*80)
print("임원 보고용 코멘트 템플릿")
print("="*80)
print("- 구조적 변화가 작을 때: '대규모 외연확장보다는 기존 활성고객 중심 초기 수용 단계로 판단됩니다.'")
print("- 단기 스파이크만 있을 때: '이벤트의 단기 관심 환기 효과는 확인되나 지속 전환은 추가 설계가 필요합니다.'")
print("- 재사용이 낮을 때: '가입 규모보다 가입 후 첫사용률/7일·30일 재사용률 중심 관리로 전환하겠습니다.'")
print("- 일부 지표가 좋을 때: '반응군에서 재사용 신호가 확인되어 타깃 중심 확장 전략을 적용하겠습니다.'")